<a href="https://colab.research.google.com/github/Impana1717/data-analysis/blob/main/AB_NYC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Step 2: Load the Datasets
# Step 2: Load the Datasets (FIXED)

nyc_df = pd.read_csv("/content/AB_NYC_2019.csv")

yt_df = pd.read_csv(
    "/content/CAvideos.csv",
    engine='python',        # Handles irregular CSV formatting
    on_bad_lines='skip',    # Skips corrupted lines
    encoding='utf-8'        # Correct YouTube CSV encoding
)

print("✅ Datasets Loaded Successfully!")
print(nyc_df.shape)
print(yt_df.shape)


✅ Datasets Loaded Successfully!
(7391, 16)
(120, 16)


In [ ]:
# Step 3: Initial Inspection
print("\n--- NYC Dataset Info ---")
nyc_df.info()

print("\n--- YouTube Dataset Info ---")
yt_df.info()

print("\n--- NYC Missing Values ---")
print(nyc_df.isnull().sum())

print("\n--- YouTube Missing Values ---")
print(yt_df.isnull().sum())


--- NYC Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7391 entries, 0 to 7390
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              7391 non-null   int64  
 1   name                            7384 non-null   object 
 2   host_id                         7391 non-null   int64  
 3   host_name                       7386 non-null   object 
 4   neighbourhood_group             7391 non-null   object 
 5   neighbourhood                   7391 non-null   object 
 6   latitude                        7391 non-null   float64
 7   longitude                       7391 non-null   float64
 8   room_type                       7390 non-null   object 
 9   price                           7390 non-null   float64
 10  minimum_nights                  7390 non-null   float64
 11  number_of_reviews               7390 non-null   float64
 12  last_rev

In [20]:
#Step 4: Handle Missing Data

# NYC Dataset
# Fill missing reviews_per_month with 0 (no reviews)
nyc_df['reviews_per_month'].fillna(0, inplace=True)

# Drop rows where host_name or name are missing (important info)
nyc_df.dropna(subset=['host_name', 'name'], inplace=True)

# YouTube Dataset
# Fill missing description with empty string
yt_df['description'].fillna("", inplace=True)

/tmp/ipython-input-3901229457.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  nyc_df['reviews_per_month'].fillna(0, inplace=True)
/tmp/ipython-input-3901229457.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [ ]:
# 🔁 Step 5: Remove Duplicates
nyc_before = nyc_df.shape[0]
nyc_df.drop_duplicates(inplace=True)
nyc_after = nyc_df.shape[0]

yt_before = yt_df.shape[0]
yt_df.drop_duplicates(inplace=True)
yt_after = yt_df.shape[0]

print(f"\n NYC Duplicates Removed: {nyc_before - nyc_after}")
print(f" YouTube Duplicates Removed: {yt_before - yt_after}")


 NYC Duplicates Removed: 0
 YouTube Duplicates Removed: 17


In [ ]:
#  Step 6: Standardization (Consistency)
nyc_df.columns = nyc_df.columns.str.lower()

yt_df.columns = yt_df.columns.str.lower()
yt_df['publish_time'] = pd.to_datetime(yt_df['publish_time'], errors='coerce')

In [ ]:
 #Step 7: Outlier Detection and Handling
Q1 = nyc_df['price'].quantile(0.25)
Q3 = nyc_df['price'].quantile(0.75)
IQR = Q3 - Q1
nyc_df = nyc_df[(nyc_df['price'] >= (Q1 - 1.5 * IQR)) & (nyc_df['price'] <= (Q3 + 1.5 * IQR))]

for col in ['views', 'likes', 'dislikes']:
    Q1 = yt_df[col].quantile(0.25)
    Q3 = yt_df[col].quantile(0.75)
    IQR = Q3 - Q1
    yt_df = yt_df[(yt_df[col] >= (Q1 - 1.5 * IQR)) & (yt_df[col] <= (Q3 + 1.5 * IQR))]

print("\n✅ Outliers handled successfully!")



✅ Outliers handled successfully!


In [ ]:
# Step 8: Verify Data Integrity
print("\n--- NYC Cleaned Data Summary ---")
print(nyc_df.describe())

print("\n--- YouTube Cleaned Data Summary ---")
print(yt_df.describe())


--- NYC Cleaned Data Summary ---
                 id       host_id     latitude    longitude        price  \
count  7.379000e+03  7.379000e+03  7379.000000  7379.000000  7378.000000   
mean   2.441921e+06  7.912539e+06    40.729249   -73.950499   167.176335   
std    1.689158e+06  7.632793e+06     0.051749     0.780323   279.188567   
min    2.539000e+03  2.571000e+03    40.508680   -74.239860    10.000000   
25%    8.210440e+05  1.598400e+06    40.688760   -73.983995    80.000000   
50%    2.223247e+06  5.072123e+06    40.723230   -73.959300   120.000000   
75%    4.025740e+06  1.197249e+07    40.763825   -73.943620   185.000000   
max    5.478741e+06  4.139808e+07    40.908040    -7.000000  9999.000000   

       minimum_nights  number_of_reviews  reviews_per_month  \
count     7378.000000        7378.000000        7379.000000   
mean         8.402548          50.846571           0.798809   
std         27.455477          72.749235           1.106206   
min          1.000000        

In [ ]:
#  Step 9: Save Cleaned Datasets
nyc_df.to_csv("/content/AB_NYC_2019_CLEANED.csv", index=False)
yt_df.to_csv("/content/CAvideos_CLEANED.csv", index=False)

print("\n✅ Cleaned datasets saved successfully!")


✅ Cleaned datasets saved successfully!
